# Edge TTS — Arabic mapping v2 (MP3 only)

Round 2: avoid harsh **سكون** endings; prefer soft vowel endings (ـه / ـي / ـا).

TTS input is **only** pronunciation variant + period.

Outputs: `generated_audio/edge_tts_mapping_v2/chosen/` and `.../comparison/`.


## 1. Dependencies

In [1]:
import subprocess
import sys

def _ensure(pkg: str, import_name: str | None = None):
    name = import_name or pkg
    try:
        __import__(name)
        print(f"[deps OK] {pkg}")
        return True
    except ImportError:
        print(f"[deps] installing {pkg} ...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])
        __import__(name)
        print(f"[deps OK] {pkg} (installed)")
        return True

_ensure("edge-tts", "edge_tts")
_ensure("nest_asyncio")
_ensure("IPython", "IPython")
print("Edge TTS dependency bootstrap done.")


[deps OK] edge-tts
[deps OK] nest_asyncio
[deps OK] IPython
Edge TTS dependency bootstrap done.


## 2. Paths and output folders

In [2]:
from pathlib import Path

BASE_DIR = Path.cwd().resolve()
AUDIO_ROOT = BASE_DIR / "generated_audio"
MAPPING_ROOT = AUDIO_ROOT / "edge_tts_mapping_v2"
DIR_CHOSEN = MAPPING_ROOT / "chosen"
DIR_COMPARISON = MAPPING_ROOT / "comparison"

DIR_NUMBERS = AUDIO_ROOT / "edge_tts_numbers"
DIR_PHRASES = AUDIO_ROOT / "edge_tts_phrases"

for d in (DIR_CHOSEN, DIR_COMPARISON, DIR_NUMBERS, DIR_PHRASES):
    d.mkdir(parents=True, exist_ok=True)

print("cwd:", BASE_DIR)
print("mapping v2:", MAPPING_ROOT)
print("chosen:", DIR_CHOSEN)
print("comparison:", DIR_COMPARISON)


cwd: C:\Users\sondo\Desktop\Voice Model Stuff
mapping v2: C:\Users\sondo\Desktop\Voice Model Stuff\generated_audio\edge_tts_mapping_v2
chosen: C:\Users\sondo\Desktop\Voice Model Stuff\generated_audio\edge_tts_mapping_v2\chosen
comparison: C:\Users\sondo\Desktop\Voice Model Stuff\generated_audio\edge_tts_mapping_v2\comparison


## 3. Egyptian letter, number, and phrase mappings

In [3]:
# Chosen: (stem, letter, pronunciation without period)
CHOSEN_MAPPINGS = [
    ("letter_ا_alif", "ا", "أَلِف"),
    ("letter_ت_teh", "ت", "تِه"),
    ("comp_seh_ث_ث_ه", "ث", "ثِه"),
    ("letter_ج_geem", "ج", "جِيم"),
    ("letter_ح_ha", "ح", "حَا"),
    ("letter_خ_kha", "خ", "خَه"),
    ("letter_ر_ra", "ر", "رَا"),
    ("letter_س_seen", "س", "سِين"),
    ("letter_ش_sheen", "ش", "شِين"),
    ("letter_ظ_dha", "ظ", "ظَه"),
    ("letter_ع_ein", "ع", "عِين"),
    ("letter_غ_ghein", "غ", "غِين"),
    ("letter_م_meem", "م", "مِيم"),
    ("letter_ن_noon", "ن", "نُون"),
    ("letter_ي_yeh", "ي", "يِه"),
    ("comp_ta_ط_ط_ه", "ط", "طَه"),
    ("comp_faa_ف_ف_ه", "ف", "فِه"),
    ("comp_ha_ه_ه_ه", "ه", "هِه"),
]

# Soft-ending variants (avoid سكون on final consonant)
COMPARISON_VARIANTS = {
    "ب": ["بِه", "بِيه", "بِي", "بَا"],
    "د": ["دَا", "دِي", "دَاه", "دَاي"],
    "ذ": ["ذَا", "ذِي", "ذَاه", "زَا"],
    "ز": ["زِي", "زَا", "زِيه", "زَاي"],
    "ص": ["صَا", "صِي", "صَاه", "صَاي"],
    "ض": ["ضَا", "ضِي", "ضَاه", "ضَاي"],
    "ق": ["قَا", "قِي", "قَاه", "قَافَا"],
    "ك": ["كَا", "كِي", "كَاه", "كَافَا"],
    "ل": ["لَا", "لِي", "لَاه", "لَاي"],
    "و": ["وَا", "وِي", "وَاه", "وِه"],
}

COMP_FILE_PREFIX = {
    "ب": "beh", "د": "dal", "ذ": "zal", "ز": "zay", "ص": "sad",
    "ض": "dad", "ق": "qaf", "ك": "kaf", "ل": "lam", "و": "waw",
}

CHOSEN_SKIP_REGEN = frozenset({"comp_faa_ف_ف_ه", "comp_ha_ه_ه_ه"})

NUMBER_PRON = {
    0: "صِفْر", 1: "وَاحِد", 2: "اِتْنِين", 3: "تَلَاتَه", 4: "أَرْبَعَه",
    5: "خَمْسَه", 6: "سِتَّه", 7: "سَبْعَه", 8: "تَمَانْيَه", 9: "تِسْعَه", 10: "عَشَرَه",
}
PHRASES = [
    ("ana_basme3ak", "أَنَا بَسْمَعَك."),
    ("mama", "مَامَا."),
    ("beit", "بَيْت."),
    ("khamsa", "خَمْسَه."),
    ("itneen", "اِتْنِين."),
    ("ahlan_beek", "أَهْلًا بِيك."),
    ("ezzayak", "إِزَّيَّك."),
    ("ayez_arooh_elbeit", "عَايِز أَرُوح البَيْت."),
]

def with_period(text: str) -> str:
    t = text.strip()
    return t if t.endswith(".") else t + "."

def number_tts(n: int) -> str:
    return with_period(NUMBER_PRON[n])

def _base_arabic_chars(text: str) -> list[str]:
    out = []
    for c in text:
        if c in "._" or c.isspace():
            continue
        if "\u0621" <= c <= "\u064a" or c in "أإآء":
            out.append(c)
    return out

def variant_path_part(pron: str) -> str:
    return "_".join(_base_arabic_chars(pron.strip().strip(".")))

def comparison_stem(letter: str, pron: str) -> str:
    prefix = COMP_FILE_PREFIX[letter]
    return f"comp_{prefix}_{letter}_{variant_path_part(pron)}"

print(f"Chosen mappings: {len(CHOSEN_MAPPINGS)}")
print(f"Comparison letters: {''.join(COMPARISON_VARIANTS)}")


Chosen mappings: 18
Comparison letters: بدذزصضقكلو


## 4. Select Edge TTS voice

In [4]:
import asyncio

import edge_tts
import nest_asyncio

nest_asyncio.apply()

PREFERRED_VOICES = ["ar-EG-SalmaNeural", "ar-EG-ShakirNeural"]
EDGE_VOICE = None
EDGE_TTS_LOADED = False
_ALL_VOICES = None

async def _list_voices():
    global _ALL_VOICES
    if _ALL_VOICES is None:
        _ALL_VOICES = await edge_tts.list_voices()
    return _ALL_VOICES

async def select_edge_voice() -> str:
    voices = await _list_voices()
    by_short = {v["ShortName"]: v for v in voices}

    for pref in PREFERRED_VOICES:
        if pref in by_short:
            return pref

    ar_names = sorted(
        v["ShortName"]
        for v in voices
        if str(v.get("Locale", "")).lower().startswith("ar")
    )
    print("Preferred voices unavailable. Available Arabic voices:")
    for name in ar_names:
        print(" ", name)

    eg = [
        v["ShortName"]
        for v in voices
        if "EG" in v.get("Locale", "") or "-EG-" in v.get("ShortName", "")
    ]
    if eg:
        return eg[0]
    if ar_names:
        return ar_names[0]
    raise RuntimeError("No Arabic Edge TTS voice found")

EDGE_VOICE = asyncio.get_event_loop().run_until_complete(select_edge_voice())
EDGE_TTS_LOADED = True
print(f"Selected Edge TTS voice: {EDGE_VOICE}")


Selected Edge TTS voice: ar-EG-SalmaNeural


## 5. Edge TTS synthesis (MP3 only)


In [5]:
import re
import time
from typing import Optional

from IPython.display import Audio as IPA, display

CHOSEN_LOG = []
COMPARISON_LOG = []
PLAYBACK_SHOWN = False

def format_bytes(nb: int) -> str:
    if nb < 1024:
        return f"{nb} B"
    if nb < 1024**2:
        return f"{nb / 1024:.2f} KB"
    return f"{nb / 1024 / 1024:.2f} MB"

async def synth_edge_mp3(text: str, mp3_path: Path) -> float:
    mp3_path.parent.mkdir(parents=True, exist_ok=True)
    t0 = time.perf_counter()
    comm = edge_tts.Communicate(text, EDGE_VOICE)
    await comm.save(str(mp3_path))
    return time.perf_counter() - t0

async def generate_mp3(
    letter: str,
    pron: str,
    mp3_path: Path,
    show_playback: bool = False,
) -> dict | None:
    global PLAYBACK_SHOWN
    tts_text = with_period(pron)
    infer_s = await synth_edge_mp3(tts_text, mp3_path)
    if not mp3_path.is_file():
        print("FAILED:", mp3_path)
        return None
    entry = {
        "letter": letter,
        "pron": pron,
        "tts_text": tts_text,
        "mp3_path": mp3_path.resolve(),
        "infer_s": infer_s,
        "bytes": mp3_path.stat().st_size,
    }
    print(f"letter: {letter}")
    print(f"variant text sent to Edge TTS: {tts_text}")
    print(f"output MP3 path: {entry['mp3_path']}")
    print(f"inference: {infer_s:.3f}s | {format_bytes(entry['bytes'])}")
    print("---")
    if show_playback:
        display(IPA(filename=str(mp3_path)))
        PLAYBACK_SHOWN = True
    return entry

def run_async(coro):
    return asyncio.get_event_loop().run_until_complete(coro)


## 6. Generate chosen mappings (MP3)


In [6]:
import shutil

CHOSEN_LOG.clear()
CHOSEN_OK = 0

print("=== Chosen mappings ===")
for stem, letter, pron in CHOSEN_MAPPINGS:
    mp3_path = DIR_CHOSEN / f"{stem}.mp3"
    comp_path = DIR_COMPARISON / f"{stem}.mp3"
    tts_text = with_period(pron)

    if mp3_path.is_file() and stem in CHOSEN_SKIP_REGEN:
        print(f"letter: {letter}")
        print(f"variant text sent to Edge TTS: {tts_text}")
        print(f"output MP3 path: {mp3_path.resolve()} (kept, skipped regen)")
        print("---")
        CHOSEN_LOG.append({
            "stem": stem, "letter": letter, "pron": pron, "tts_text": tts_text,
            "mp3_path": mp3_path.resolve(), "infer_s": 0.0, "bytes": mp3_path.stat().st_size,
        })
        CHOSEN_OK += 1
        continue

    if stem in CHOSEN_SKIP_REGEN and comp_path.is_file() and not mp3_path.is_file():
        shutil.copy2(comp_path, mp3_path)
        print(f"letter: {letter}")
        print(f"variant text sent to Edge TTS: {tts_text}")
        print(f"output MP3 path: {mp3_path.resolve()} (copied from comparison)")
        print("---")
        CHOSEN_LOG.append({
            "stem": stem, "letter": letter, "pron": pron, "tts_text": tts_text,
            "mp3_path": mp3_path.resolve(), "infer_s": 0.0, "bytes": mp3_path.stat().st_size,
        })
        CHOSEN_OK += 1
        continue

    if mp3_path.is_file():
        print(f"letter: {letter}")
        print(f"variant text sent to Edge TTS: {tts_text}")
        print(f"output MP3 path: {mp3_path.resolve()} (kept)")
        print("---")
        CHOSEN_LOG.append({
            "stem": stem, "letter": letter, "pron": pron, "tts_text": tts_text,
            "mp3_path": mp3_path.resolve(), "infer_s": 0.0, "bytes": mp3_path.stat().st_size,
        })
        CHOSEN_OK += 1
        continue

    entry = run_async(generate_mp3(letter, pron, mp3_path, show_playback=False))
    if entry:
        entry["stem"] = stem
        CHOSEN_LOG.append(entry)
        CHOSEN_OK += 1

print(f"Chosen: {CHOSEN_OK}/{len(CHOSEN_MAPPINGS)} MP3 files")


=== Chosen mappings ===
letter: ا
variant text sent to Edge TTS: أَلِف.
output MP3 path: C:\Users\sondo\Desktop\Voice Model Stuff\generated_audio\edge_tts_mapping_v2\chosen\letter_ا_alif.mp3 (kept)
---
letter: ت
variant text sent to Edge TTS: تِه.
output MP3 path: C:\Users\sondo\Desktop\Voice Model Stuff\generated_audio\edge_tts_mapping_v2\chosen\letter_ت_teh.mp3 (kept)
---
letter: ث
variant text sent to Edge TTS: ثِه.
output MP3 path: C:\Users\sondo\Desktop\Voice Model Stuff\generated_audio\edge_tts_mapping_v2\chosen\comp_seh_ث_ث_ه.mp3 (kept)
---
letter: ج
variant text sent to Edge TTS: جِيم.
output MP3 path: C:\Users\sondo\Desktop\Voice Model Stuff\generated_audio\edge_tts_mapping_v2\chosen\letter_ج_geem.mp3 (kept)
---
letter: ح
variant text sent to Edge TTS: حَا.
output MP3 path: C:\Users\sondo\Desktop\Voice Model Stuff\generated_audio\edge_tts_mapping_v2\chosen\letter_ح_ha.mp3 (kept)
---
letter: خ
variant text sent to Edge TTS: خَه.
output MP3 path: C:\Users\sondo\Desktop\Voice Mod

## 7. Comparison variants (soft endings — avoid سكون)

Letters not in chosen. No ا / أ. MP3 only. Replaces prior comparison clips.


In [7]:
removed = 0
for old in DIR_COMPARISON.glob("*.mp3"):
    old.unlink()
    removed += 1
print(f"Cleared {removed} old comparison MP3(s)")

COMPARISON_LOG.clear()
COMP_OK = 0
COMP_TOTAL = sum(len(v) for v in COMPARISON_VARIANTS.values())

for letter, variants in COMPARISON_VARIANTS.items():
    print(f"\n=== {letter} variants ===")
    for pron in variants:
        stem = comparison_stem(letter, pron)
        mp3_path = DIR_COMPARISON / f"{stem}.mp3"
        entry = run_async(generate_mp3(letter, pron, mp3_path, show_playback=False))
        if entry:
            entry["stem"] = stem
            COMPARISON_LOG.append(entry)
            COMP_OK += 1

COMPARISON_OK = COMP_OK == COMP_TOTAL and COMP_OK > 0
print(f"\nComparison: {COMP_OK}/{COMP_TOTAL} MP3 files")


Cleared 40 old comparison MP3(s)

=== ب variants ===


letter: ب
variant text sent to Edge TTS: بِه.
output MP3 path: C:\Users\sondo\Desktop\Voice Model Stuff\generated_audio\edge_tts_mapping_v2\comparison\comp_beh_ب_ب_ه.mp3
inference: 0.601s | 8.72 KB
---


letter: ب
variant text sent to Edge TTS: بِيه.
output MP3 path: C:\Users\sondo\Desktop\Voice Model Stuff\generated_audio\edge_tts_mapping_v2\comparison\comp_beh_ب_ب_ي_ه.mp3
inference: 0.619s | 7.88 KB
---


letter: ب
variant text sent to Edge TTS: بِي.
output MP3 path: C:\Users\sondo\Desktop\Voice Model Stuff\generated_audio\edge_tts_mapping_v2\comparison\comp_beh_ب_ب_ي.mp3
inference: 0.545s | 7.59 KB
---


letter: ب
variant text sent to Edge TTS: بَا.
output MP3 path: C:\Users\sondo\Desktop\Voice Model Stuff\generated_audio\edge_tts_mapping_v2\comparison\comp_beh_ب_ب_ا.mp3
inference: 0.552s | 7.31 KB
---

=== د variants ===


...

letter: ل
variant text sent to Edge TTS: لَاي.
output MP3 path: C:\Users\sondo\Desktop\Voice Model Stuff\generated_audio\edge_tts_mapping_v2\comparison\comp_lam_ل_ل_ا_ي.mp3
inference: 0.790s | 8.44 KB
---

=== و variants ===


letter: و
variant text sent to Edge TTS: وَا.
output MP3 path: C:\Users\sondo\Desktop\Voice Model Stuff\generated_audio\edge_tts_mapping_v2\comparison\comp_waw_و_و_ا.mp3
inference: 0.631s | 7.59 KB
---


letter: و
variant text sent to Edge TTS: وِي.
output MP3 path: C:\Users\sondo\Desktop\Voice Model Stuff\generated_audio\edge_tts_mapping_v2\comparison\comp_waw_و_و_ي.mp3
inference: 0.822s | 7.17 KB
---


letter: و
variant text sent to Edge TTS: وَاه.
output MP3 path: C:\Users\sondo\Desktop\Voice Model Stuff\generated_audio\edge_tts_mapping_v2\comparison\comp_waw_و_و_ا_ه.mp3
inference: 0.607s | 9.42 KB
---


letter: و
variant text sent to Edge TTS: وِه.
output MP3 path: C:\Users\sondo\Desktop\Voice Model Stuff\generated_audio\edge_tts_mapping_v2\comparison\comp_waw_و_و_ه.mp3
inference: 0.507s | 8.02 KB
---

Comparison: 40/40 MP3 files


## 8. Playback — new comparison variants


In [8]:
def _play_entry(entry: dict, title: str = ""):
    if title:
        print(title)
    print(f"  letter: {entry['letter']}")
    print(f"  variant text sent to Edge TTS: {entry['tts_text']}")
    print(f"  output MP3 path: {entry['mp3_path']}")
    display(IPA(filename=str(entry["mp3_path"])))
    print()

by_letter = {}
for e in COMPARISON_LOG:
    by_letter.setdefault(e["letter"], []).append(e)

print("=== New comparison variants (soft endings) ===")
for letter in COMPARISON_VARIANTS:
    print(f"--- {letter} ---")
    for entry in by_letter.get(letter, []):
        _play_entry(entry)

PLAYBACK_SHOWN = True


=== New comparison variants (soft endings) ===
--- ب ---
  letter: ب
  variant text sent to Edge TTS: بِه.
  output MP3 path: C:\Users\sondo\Desktop\Voice Model Stuff\generated_audio\edge_tts_mapping_v2\comparison\comp_beh_ب_ب_ه.mp3



  letter: ب
  variant text sent to Edge TTS: بِيه.
  output MP3 path: C:\Users\sondo\Desktop\Voice Model Stuff\generated_audio\edge_tts_mapping_v2\comparison\comp_beh_ب_ب_ي_ه.mp3



  letter: ب
  variant text sent to Edge TTS: بِي.
  output MP3 path: C:\Users\sondo\Desktop\Voice Model Stuff\generated_audio\edge_tts_mapping_v2\comparison\comp_beh_ب_ب_ي.mp3



  letter: ب
  variant text sent to Edge TTS: بَا.
  output MP3 path: C:\Users\sondo\Desktop\Voice Model Stuff\generated_audio\edge_tts_mapping_v2\comparison\comp_beh_ب_ب_ا.mp3



--- د ---
  letter: د
  variant text sent to Edge TTS: دَا.
  output MP3 path: C:\Users\sondo\Desktop\Voice Model Stuff\generated_audio\edge_tts_mapping_v2\comparison\comp_dal_د_د_ا.mp3


...


  letter: ل
  variant text sent to Edge TTS: لَاي.
  output MP3 path: C:\Users\sondo\Desktop\Voice Model Stuff\generated_audio\edge_tts_mapping_v2\comparison\comp_lam_ل_ل_ا_ي.mp3



--- و ---
  letter: و
  variant text sent to Edge TTS: وَا.
  output MP3 path: C:\Users\sondo\Desktop\Voice Model Stuff\generated_audio\edge_tts_mapping_v2\comparison\comp_waw_و_و_ا.mp3



  letter: و
  variant text sent to Edge TTS: وِي.
  output MP3 path: C:\Users\sondo\Desktop\Voice Model Stuff\generated_audio\edge_tts_mapping_v2\comparison\comp_waw_و_و_ي.mp3



  letter: و
  variant text sent to Edge TTS: وَاه.
  output MP3 path: C:\Users\sondo\Desktop\Voice Model Stuff\generated_audio\edge_tts_mapping_v2\comparison\comp_waw_و_و_ا_ه.mp3



  letter: و
  variant text sent to Edge TTS: وِه.
  output MP3 path: C:\Users\sondo\Desktop\Voice Model Stuff\generated_audio\edge_tts_mapping_v2\comparison\comp_waw_و_و_ه.mp3


## 9. Number tests (MP3, optional)


In [9]:
NUMBERS_LOG = []
ok_num = 0

for n in range(0, 11):
    tts_text = number_tts(n)
    mp3_path = DIR_NUMBERS / f"num_{n:02d}.mp3"
    entry = run_async(generate_mp3(str(n), NUMBER_PRON[n], mp3_path))
    if entry:
        NUMBERS_LOG.append(entry)
        ok_num += 1

NUMBERS_OK = ok_num == 11
print(f"Numbers: {ok_num}/11 MP3 files")


letter: 0
variant text sent to Edge TTS: صِفْر.
output MP3 path: C:\Users\sondo\Desktop\Voice Model Stuff\generated_audio\edge_tts_numbers\num_00.mp3
inference: 0.654s | 8.30 KB
---


letter: 1
variant text sent to Edge TTS: وَاحِد.
output MP3 path: C:\Users\sondo\Desktop\Voice Model Stuff\generated_audio\edge_tts_numbers\num_01.mp3
inference: 0.616s | 9.00 KB
---


letter: 2
variant text sent to Edge TTS: اِتْنِين.
output MP3 path: C:\Users\sondo\Desktop\Voice Model Stuff\generated_audio\edge_tts_numbers\num_02.mp3
inference: 0.558s | 9.98 KB
---


letter: 3
variant text sent to Edge TTS: تَلَاتَه.
output MP3 path: C:\Users\sondo\Desktop\Voice Model Stuff\generated_audio\edge_tts_numbers\num_03.mp3
inference: 0.616s | 9.98 KB
---


letter: 4
variant text sent to Edge TTS: أَرْبَعَه.
output MP3 path: C:\Users\sondo\Desktop\Voice Model Stuff\generated_audio\edge_tts_numbers\num_04.mp3
inference: 0.610s | 9.70 KB
---


...

letter: 6
variant text sent to Edge TTS: سِتَّه.
output MP3 path: C:\Users\sondo\Desktop\Voice Model Stuff\generated_audio\edge_tts_numbers\num_06.mp3
inference: 0.627s | 9.42 KB
---


letter: 7
variant text sent to Edge TTS: سَبْعَه.
output MP3 path: C:\Users\sondo\Desktop\Voice Model Stuff\generated_audio\edge_tts_numbers\num_07.mp3
inference: 0.646s | 9.70 KB
---


letter: 8
variant text sent to Edge TTS: تَمَانْيَه.
output MP3 path: C:\Users\sondo\Desktop\Voice Model Stuff\generated_audio\edge_tts_numbers\num_08.mp3
inference: 0.631s | 10.12 KB
---


letter: 9
variant text sent to Edge TTS: تِسْعَه.
output MP3 path: C:\Users\sondo\Desktop\Voice Model Stuff\generated_audio\edge_tts_numbers\num_09.mp3
inference: 0.678s | 9.42 KB
---


letter: 10
variant text sent to Edge TTS: عَشَرَه.
output MP3 path: C:\Users\sondo\Desktop\Voice Model Stuff\generated_audio\edge_tts_numbers\num_10.mp3
inference: 0.603s | 10.27 KB
---
Numbers: 11/11 MP3 files


## 10. Egyptian Arabic phrase tests (MP3, optional)


In [10]:
PHRASES_LOG = []
ok_phrase = 0

for slug, phrase in PHRASES:
    tts_text = with_period(phrase)
    mp3_path = DIR_PHRASES / f"{slug}.mp3"
    entry = run_async(generate_mp3(slug, phrase.rstrip("."), mp3_path))
    if entry:
        PHRASES_LOG.append(entry)
        ok_phrase += 1

PHRASES_OK = ok_phrase == len(PHRASES)
print(f"Phrases: {ok_phrase}/{len(PHRASES)} MP3 files")


letter: ana_basme3ak
variant text sent to Edge TTS: أَنَا بَسْمَعَك.
output MP3 path: C:\Users\sondo\Desktop\Voice Model Stuff\generated_audio\edge_tts_phrases\ana_basme3ak.mp3
inference: 0.653s | 11.67 KB
---


letter: mama
variant text sent to Edge TTS: مَامَا.
output MP3 path: C:\Users\sondo\Desktop\Voice Model Stuff\generated_audio\edge_tts_phrases\mama.mp3
inference: 0.540s | 8.72 KB
---


letter: beit
variant text sent to Edge TTS: بَيْت.
output MP3 path: C:\Users\sondo\Desktop\Voice Model Stuff\generated_audio\edge_tts_phrases\beit.mp3
inference: 0.636s | 8.02 KB
---


letter: khamsa
variant text sent to Edge TTS: خَمْسَه.
output MP3 path: C:\Users\sondo\Desktop\Voice Model Stuff\generated_audio\edge_tts_phrases\khamsa.mp3
inference: 0.589s | 9.56 KB
---


letter: itneen
variant text sent to Edge TTS: اِتْنِين.
output MP3 path: C:\Users\sondo\Desktop\Voice Model Stuff\generated_audio\edge_tts_phrases\itneen.mp3
inference: 0.618s | 9.98 KB
---


letter: ahlan_beek
variant text sent to Edge TTS: أَهْلًا بِيك.
output MP3 path: C:\Users\sondo\Desktop\Voice Model Stuff\generated_audio\edge_tts_phrases\ahlan_beek.mp3
inference: 0.659s | 10.27 KB
---


letter: ezzayak
variant text sent to Edge TTS: إِزَّيَّك.
output MP3 path: C:\Users\sondo\Desktop\Voice Model Stuff\generated_audio\edge_tts_phrases\ezzayak.mp3
inference: 0.694s | 10.12 KB
---


letter: ayez_arooh_elbeit
variant text sent to Edge TTS: عَايِز أَرُوح البَيْت.
output MP3 path: C:\Users\sondo\Desktop\Voice Model Stuff\generated_audio\edge_tts_phrases\ayez_arooh_elbeit.mp3
inference: 0.639s | 13.92 KB
---
Phrases: 8/8 MP3 files


## 11. Output summary


In [11]:
chosen_mp3 = sorted(DIR_CHOSEN.glob("*.mp3"))
comp_mp3 = sorted(DIR_COMPARISON.glob("*.mp3"))

print("Chosen folder:", DIR_CHOSEN)
for p in chosen_mp3:
    print(" ", p.name, format_bytes(p.stat().st_size))

print("\nComparison folder:", DIR_COMPARISON)
for p in comp_mp3:
    print(" ", p.name, format_bytes(p.stat().st_size))

print(f"\nTotal chosen: {len(chosen_mp3)} (expected {len(CHOSEN_MAPPINGS)})")
print(f"Total comparison: {len(comp_mp3)} (expected {COMP_TOTAL})")


Chosen folder: C:\Users\sondo\Desktop\Voice Model Stuff\generated_audio\edge_tts_mapping_v2\chosen
  comp_faa_ف_ف_ه.mp3 7.31 KB
  comp_ha_ه_ه_ه.mp3 8.30 KB
  comp_seh_ث_ث_ه.mp3 7.17 KB
  comp_ta_ط_ط_ه.mp3 8.16 KB
  letter_ا_alif.mp3 8.44 KB
  letter_ت_teh.mp3 8.02 KB
  letter_ج_geem.mp3 9.14 KB
  letter_ح_ha.mp3 7.88 KB
  letter_خ_kha.mp3 8.02 KB
  letter_ر_ra.mp3 7.73 KB
  letter_س_seen.mp3 9.28 KB
  letter_ش_sheen.mp3 9.42 KB
  letter_ظ_dha.mp3 8.44 KB
  letter_ع_ein.mp3 9.70 KB
  letter_غ_ghein.mp3 9.28 KB
  letter_م_meem.mp3 8.58 KB
  letter_ن_noon.mp3 8.58 KB
  letter_ي_yeh.mp3 7.45 KB

Comparison folder: C:\Users\sondo\Desktop\Voice Model Stuff\generated_audio\edge_tts_mapping_v2\comparison
  comp_beh_ب_ب_ا.mp3 7.31 KB
  comp_beh_ب_ب_ه.mp3 8.72 KB
  comp_beh_ب_ب_ي.mp3 7.59 KB
  comp_beh_ب_ب_ي_ه.mp3 7.88 KB
  comp_dad_ض_ض_ا.mp3 7.59 KB
  comp_dad_ض_ض_ا_ه.mp3 9.00 KB
  comp_dad_ض_ض_ا_ي.mp3 8.44 KB
  comp_dad_ض_ض_ي.mp3 8.02 KB
  comp_dal_د_د_ا.mp3 7.59 KB
  comp_dal_د_د_ا_ه.mp3 8.16

## 12. Final verification


In [12]:
checks = [
    ("Edge TTS loaded", EDGE_TTS_LOADED and bool(EDGE_VOICE)),
    ("Chosen MP3 generated", CHOSEN_OK == len(CHOSEN_MAPPINGS)),
    ("Comparison MP3 generated", COMPARISON_OK),
    ("Playback displayed", PLAYBACK_SHOWN),
]

for label, ok in checks:
    mark = "OK" if ok else "FAIL"
    print(f"[{mark}] {label}")

ALL_OK = all(ok for _, ok in checks)
print("\nVoice:", EDGE_VOICE)
print("mapping v2:", MAPPING_ROOT)


[OK] Edge TTS loaded
[OK] Chosen MP3 generated
[OK] Comparison MP3 generated
[OK] Playback displayed

Voice: ar-EG-SalmaNeural
mapping v2: C:\Users\sondo\Desktop\Voice Model Stuff\generated_audio\edge_tts_mapping_v2
